# pass2 verification

Our own L4 production is run over the pass2 **L3** files, and every variable is
compared **event by event** with the values the production itself wrote into the
pass2 **L4** files.

The production publishes an HDF5 beside every L4 `.i3.zst`, same stem, one per
file -- that is the answer key, read in place.  Nothing here writes to it.

What makes a result trustworthy here:

- **One L3 file per HDF5.**  `(Run, Event, SubEvent)` is unique only within a
  single L3 file, so a chunked production could not be matched.  `pass2.match()`
  refuses rather than producing a plausible-looking table that is really an
  event-mixing artefact.
- **Read the control rows first.**  Some rows run the ORIGINAL IceTray modules
  on our side too (`iLineFit_speed`, the hit statistics) or come straight from
  L3 (`NchCleaned`, `ICVetoHits`).  If one of those disagrees, the two sides did
  not see the same input and nothing below it means anything.
- **Each sample uses the GCD from its own directory.**  Every geometry-derived
  variable comes out of it.

## 0. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, glob, subprocess, collections
import numpy as np

# Find the repository root: the directory holding `oscnext_l4`.
ROOT = os.environ.get("OSCNEXT_L4_ROOT")
if not ROOT:
    d = os.getcwd()
    while d != "/":
        if os.path.isdir(os.path.join(d, "oscnext_l4")):
            ROOT = d
            break
        d = os.path.dirname(d)
if not ROOT:
    raise SystemExit(
        "Repository root not found.  Start Jupyter from the repository "
        "directory, or set OSCNEXT_L4_ROOT.  Note that a kernel restart does "
        "NOT change the working directory -- that comes from the server.")
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print("root   :", ROOT)
print("python :", sys.executable)

from verification import pass2 as P
print("samples:", ", ".join(sorted(P.PASS2_L3)))

## 1. Configuration

`N_FILES` is per sample.  Start small: one file is already ~8000 events, which
is enough to see any real disagreement.  Raise it once the table is clean.

In [ ]:
SAMPLES = ["NuE", "NuMu", "NuTau", "MuonGun", "Noise"]
N_FILES = 2                      # per sample
OUTDIR  = "L4_output/pass2_check"

# Extra process_L4.py flags.  Empty is right: the defaults now reproduce pass2
# for micro_count and accumulated_time.  Use ["--micro-count-cleaned"] or
# ["--accumulated-time-note"] to check the technical note's reading instead.
EXTRA_FLAGS = []

os.makedirs(OUTDIR, exist_ok=True)
print("%d sample(s) x %d file(s) -> %s" % (len(SAMPLES), N_FILES, OUTDIR))

## 2. The file pairs

For every L3 file: its L4 partner, the production HDF5 beside it, and the GCD
from that directory.  Not every L3 file has an L4 partner -- orphans are
reported rather than skipped quietly.

In [ ]:
def production_hdf5(l4_path):
    """The HDF5 the production published next to an L4 .i3 file."""
    for ext in (".i3.zst", ".i3.bz2", ".i3.gz", ".i3"):
        if l4_path.endswith(ext):
            return l4_path[: -len(ext)] + ".hdf5"
    return None


_gcd_cache = {}

def find_gcd(*paths):
    """The GCD that sits with the files -- never one passed in by hand."""
    for path in paths:
        d = os.path.dirname(path)
        if d in _gcd_cache:
            return _gcd_cache[d]
        hits = sorted(glob.glob(os.path.join(d, "GeoCalibDetectorStatus*.i3*")))
        if len(hits) == 1:
            _gcd_cache[d] = hits[0]
            return hits[0]
        if len(hits) > 1:
            raise RuntimeError("%d GCDs in %s: %s" % (len(hits), d,
                               ", ".join(map(os.path.basename, hits))))
    raise RuntimeError("no GCD beside " + " or ".join(paths))


JOBS = []            # (sample, tag, l3, production_hdf5, gcd, ours_hdf5)

for sample in SAMPLES:
    pats = P.PASS2_L3[sample]
    if isinstance(pats, str):
        pats = [pats]
    l3 = sorted(f for p in pats for f in glob.glob(p))
    pairs, orphans = P.pair_files(l3[: N_FILES] if N_FILES else l3)

    kept = 0
    for f3, f4 in pairs:
        ref = production_hdf5(f4)
        if not (ref and os.path.exists(ref)):
            continue
        tag = os.path.basename(f3).replace(".i3.zst", "").replace(".i3", "")
        JOBS.append((sample, tag, f3, ref, find_gcd(f4, f3),
                     os.path.join(OUTDIR, "ours_%s.hdf5" % tag)))
        kept += 1

    print("%-8s %3d L3 file(s), %3d paired, %3d with a production hdf5%s"
          % (sample, len(l3[: N_FILES] if N_FILES else l3), len(pairs), kept,
             ("   [%d orphan]" % len(orphans)) if orphans else ""))

print()
for s in SAMPLES:
    g = {j[4] for j in JOBS if j[0] == s}
    for gcd in sorted(g):
        print("%-8s GCD %s" % (s, os.path.basename(gcd)))
print("\n%d job(s) in total" % len(JOBS))

## 3. Produce our side

One tray run per L3 file.  Files already produced are skipped, so re-running
this cell after a code change means deleting the ones you want rebuilt (or
setting `REBUILD = True`).

`WORKERS` runs that many trays at once.  Each is a separate process writing its
own file with no shared state, so the speedup is close to linear -- but **cobalt
is a shared machine**: 4-8 is neighbourly, 64 is not.

In [ ]:
import sys
REBUILD = False

def produce(job, extra_flags=EXTRA_FLAGS, quiet=True):
    sample, tag, f3, ref, gcd, ours = job
    if os.path.exists(ours) and not REBUILD:
        return "skipped"
    # Clear what a killed or earlier run left (process_L4.py's --overwrite
    # applies to --chunk-files parts only).  The sidecar is <output>.meta.json
    # and an unfinished output carries the _incomplete_ prefix.
    _d, _b = os.path.split(ours)
    for stale in (ours, ours + ".badfiles.txt", ours + ".meta.json",
                  os.path.join(_d, "_incomplete_" + _b)):
        if os.path.exists(stale):
            os.remove(stale)
    cmd = [sys.executable, "scripts/process_L4.py", "--gcd", gcd, "--input", f3,
           "--cleaned-pulses", P.PASS2_CLEANED_PULSES,
           "--output-hdf5", ours, "--scan", "off"]
    cmd += list(P.PASS2_FLAGS.get(sample, [])) + list(extra_flags)
    r = subprocess.run(cmd, capture_output=quiet, text=True)
    if r.returncode != 0:
        if quiet and r.stderr:
            print(r.stderr[-2000:])
        return "FAILED"
    return "produced"


WORKERS = 4                      # 1 = sequential

import time
import threading
from concurrent.futures import ThreadPoolExecutor

# Threads, not processes: each worker only waits on a subprocess, so the GIL is
# released the whole time and the tray runs are genuinely concurrent.
_print_lock = threading.Lock()
counts = collections.Counter()
t0 = time.time()

def _one(item):
    i, job = item
    status = produce(job)
    with _print_lock:
        counts[status] += 1
        print("[%3d/%3d] %5.0fs %-8s %-46s %s"
              % (sum(counts.values()), len(JOBS), time.time() - t0,
                 job[0], job[1], status), flush=True)
    return status

if WORKERS > 1:
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        list(pool.map(_one, enumerate(JOBS, 1)))
else:
    for item in enumerate(JOBS, 1):
        _one(item)

print()
print(dict(counts), " %.0f s total" % (time.time() - t0))

## 4. Compare, pooled over every file

`pass2.load_comparison` matches the events through hdfwriter's `__I3Index__`
tables, then the two sides are stacked so each variable is judged on ALL events
at once rather than file by file.

In [ ]:
TABLE = P.compare_table(P.PASS2_CLEANED_PULSES)
NAMES = [row[0] for row in TABLE]
REWRITTEN = {row[0]: row[3] for row in TABLE}

pool = {n: {"ours": [], "pass2": [], "sample": []} for n in NAMES}
per_file = []
problems = []

for sample, tag, f3, ref, gcd, ours in JOBS:
    if not os.path.exists(ours):
        continue
    try:
        data, missing, cnt = P.load_comparison(ours, ref,
                                               P.PASS2_CLEANED_PULSES)
    except Exception as exc:
        problems.append((sample, tag, "%s: %s" % (type(exc).__name__, exc)))
        continue

    per_file.append((sample, tag, cnt["ours"], cnt["pass2"], cnt["matched"]))
    for m in missing:
        problems.append((sample, tag, "missing %s" % (m,)))
    for n in NAMES:
        o, p = data["ours"][n], data["pass2"][n]
        pool[n]["ours"].append(o)
        pool[n]["pass2"].append(p)
        pool[n]["sample"].append(np.full(o.size, sample, dtype=object))

for n in NAMES:
    for k in ("ours", "pass2", "sample"):
        pool[n][k] = (np.concatenate(pool[n][k]) if pool[n][k]
                      else np.zeros(0))

print("%-8s %-52s %8s %8s %8s" % ("sample", "file", "ours", "pass2", "matched"))
for row in per_file:
    print("%-8s %-52s %8d %8d %8d" % row)

n_matched = sum(r[4] for r in per_file)
print("\n%d file(s), %d matched events" % (len(per_file), n_matched))

if problems:
    print("\nPROBLEMS (%d):" % len(problems))
    for s, t, msg in problems[:20]:
        print("  %-8s %-44s %s" % (s, t, msg))

## 5. The summary table

`agree` is the FRACTION of events that match, not the worst event -- deciding
it on the maximum once stamped a variable `DIFFERS` that agreed in 99.85% of
cases.  Integers are held to exact equality; floats to `1e-6` relative.

In [ ]:
SUMMARY = []
for n in NAMES:
    o, p = pool[n]["ours"], pool[n]["pass2"]
    if o.size == 0:
        SUMMARY.append((n, REWRITTEN[n], "no data", 0, float("nan"),
                        float("nan")))
        continue
    status, cnt, agree, max_abs, max_rel = P._verdict(n, o, p)
    SUMMARY.append((n, REWRITTEN[n], status, cnt, agree, max_abs))

hdr = "%-22s %-10s %-10s %9s %9s %14s"
print(hdr % ("variable", "kind", "verdict", "n", "agree", "max |diff|"))
print("-" * 80)
for n, rew, status, cnt, agree, max_abs in SUMMARY:
    kind = "rewritten" if rew else ("derived" if n in P.DEPENDS_ON
                                    else "control")
    print(hdr % (n, kind, status, cnt,
                 "%.4f" % agree if agree == agree else "-",
                 "%.6g" % max_abs if max_abs == max_abs else "-"))

bad = [s for s in SUMMARY if s[2] not in ("identical", "agrees")]
print()
if not bad:
    print("Every row agrees.")
else:
    print("%d row(s) not clean:" % len(bad))
    for n, rew, status, cnt, agree, _ in bad:
        note = P.EXPECTED_DEVIATION.get(n) or P.DEPENDS_ON.get(n)
        print("  %-22s %-8s %.4f%s" % (n, status, agree,
                                       ("  <- " + note.split(".")[0])
                                       if note else ""))

## 6. Agreement per variable

Status colour plus the verdict in text -- identity is never colour alone.  The
palette was validated for colour-vision deficiency (worst adjacent pair
dE 25.7 under deuteranopia); red and amber, the obvious choice, are
indistinguishable there.

In [ ]:
import matplotlib.pyplot as plt

INK      = "#24292f"
MUTED    = "#6e7781"
GRID     = "#d8dee4"
STATUS   = {"identical": "#1a7f37", "agrees": "#1a7f37",
            "mostly": "#0969da", "DIFFERS": "#cf222e",
            "no overlap": "#6e7781", "no data": "#6e7781"}

rows = [s for s in SUMMARY if s[3] > 0]
rows = sorted(rows, key=lambda r: r[4])          # worst at the bottom

fig, ax = plt.subplots(figsize=(8.5, 0.42 * len(rows) + 1.4))
y = np.arange(len(rows))
ax.barh(y, [r[4] for r in rows], height=0.6,
        color=[STATUS.get(r[2], MUTED) for r in rows])

ax.set_yticks(y)
ax.set_yticklabels(["%s%s" % (r[0], "" if r[1] else "  (control)")
                    for r in rows], fontsize=9)
ax.set_xlim(0, 1.0)
ax.set_xlabel("fraction of events agreeing", color=MUTED, fontsize=9)
ax.set_title("Our L4 variables against the pass2 production, %d events"
             % n_matched, loc="left", color=INK, fontsize=11, pad=12)

for spine in ("top", "right", "left"):
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRID)
ax.tick_params(colors=MUTED, length=0)
ax.xaxis.grid(True, color=GRID, linewidth=0.6)
ax.set_axisbelow(True)

for yi, r in zip(y, rows):
    ax.text(min(r[4], 0.985) + 0.012, yi, "%s  %.3f" % (r[2], r[4]),
            va="center", ha="left", fontsize=8, color=INK)

fig.tight_layout()
plt.show()

## 7. Where the differences are

One panel per variable that is not identical: the distribution of
`ours - pass2`.  A single spike at zero with a thin tail is a handful of
disagreeing events; a shifted or broad distribution is a different definition.

In [ ]:
interesting = [n for n, _, status, cnt, _, _ in SUMMARY
               if cnt > 0 and status != "identical"]

if not interesting:
    print("Every variable is bitwise identical -- nothing to plot.")
else:
    ncol = 3
    nrow = int(np.ceil(len(interesting) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.0 * ncol, 2.6 * nrow))
    axes = np.atleast_1d(axes).ravel()

    for ax, name in zip(axes, interesting):
        o, p = pool[name]["ours"], pool[name]["pass2"]
        both = np.isfinite(o) & np.isfinite(p)
        d = (o - p)[both]
        nz = int((d != 0).sum())

        ax.hist(d, bins=41, color="#0969da")
        ax.set_title("%s\n%d of %d events differ" % (name, nz, d.size),
                     fontsize=9, color=INK, loc="left")
        ax.set_xlabel("ours - pass2", fontsize=8, color=MUTED)
        ax.set_yscale("log")
        for spine in ("top", "right"):
            ax.spines[spine].set_visible(False)
        for spine in ("left", "bottom"):
            ax.spines[spine].set_color(GRID)
        ax.tick_params(colors=MUTED, labelsize=8)
        ax.yaxis.grid(True, color=GRID, linewidth=0.6)
        ax.set_axisbelow(True)

    for ax in axes[len(interesting):]:
        ax.set_visible(False)
    fig.tight_layout()
    plt.show()

## 8. Drill into one variable

The events where a variable disagrees, with the sample they came from.  Use
this to tell "a handful of odd events" apart from "a systematic shift".

In [ ]:
VAR = "accumulated_time"          # change this

o, p = pool[VAR]["ours"], pool[VAR]["pass2"]
s = pool[VAR]["sample"]
both = np.isfinite(o) & np.isfinite(p)
d = o - p
bad = both & (d != 0)

print("%s: %d of %d events differ (%.3f%%)"
      % (VAR, int(bad.sum()), int(both.sum()),
         100.0 * bad.sum() / max(int(both.sum()), 1)))

if bad.any():
    print("\nby sample:")
    for name in sorted(set(s[bad])):
        m = bad & (s == name)
        print("   %-8s %6d of %6d" % (name, int(m.sum()),
                                      int((both & (s == name)).sum())))

    print("\npass2 stored exactly 0 in %d of them"
          % int((bad & (p == 0)).sum()))

    order = np.argsort(-np.abs(np.where(bad, d, 0)))[:15]
    print("\n%-8s %14s %14s %14s" % ("sample", "ours", "pass2", "diff"))
    for i in order:
        if not bad[i]:
            break
        print("%-8s %14.6g %14.6g %14.6g" % (s[i], o[i], p[i], d[i]))